# 18 · Back up an experiment's outputs to private Hub repos

Collects a finished experiment's outputs from the attached kernel outputs and uploads them to **private** Hub repos, one kind of artifact per repo: LoRA adapters in **model** repos (one per fine-tune condition, adapters only), the probe bundle and the logs in **dataset** repos. Writes a card per repo and a SHA-256 manifest, uploads resumably, then verifies every file against its checksum. **Run from the Kaggle editor with the `HF_TOKEN` secret ticked.** Harmful text is never uploaded: circuit-validation samples are skipped and example answers are stripped.

| | |
|---|---|
| **Kaggle settings** | Accelerator: None (CPU is enough) · Internet: **On** · Secrets: `HF_TOKEN` |
| **Attach as inputs** | `safetypulse-lib` (sp_utils.py) · outputs of the experiment's kernels (not the model mirror) |
| **Reads** | everything attached |
| **Writes** (`/kaggle/working/`) | `hub_stage/<repo>/` (what was uploaded), `hub_stage/upload_receipt.json` |

Keep this notebook and its outputs **private**: they contain harmful prompts.
After it finishes: *Save Version → Save & Run All*, then download the executed notebook into
`kaggle/executed/` and attach this notebook's output to the next notebook.

In [1]:
# Setup: find sp_utils.py (attach the `safetypulse-lib` dataset: see kaggle/README.md)
import glob, os, sys
_found = (glob.glob("/kaggle/input/**/sp_utils.py", recursive=True)
          + glob.glob("/kaggle/usr/lib/**/sp_utils.py", recursive=True)
          + glob.glob("../lib/sp_utils.py"))
assert _found, "sp_utils.py not found. Attach the safetypulse-lib dataset (Add Input) and re-run."
sys.path.insert(0, os.path.dirname(os.path.abspath(_found[0])))
import sp_utils as sp
sp.env_info()

{'torch': '2.11.0+cpu',
 'transformers': '5.20.2',
 'cuda_available': False,
 'on_kaggle': True,
 'smoke': False,
 'peft': '0.21.2'}

In [2]:
import hashlib, json, shutil, time
from pathlib import Path

OWNER = os.environ.get("SP_HUB_OWNER", "purvansh01")
NAME = os.environ.get("SP_HUB_NAME", "safetypulse-earth")            # repo name prefix
EXPERIMENT = os.environ.get("SP_EXPERIMENT", "H1: transfer to Tiny Aya Earth (docs/13_PREREGISTRATION.md)")
BASE = os.environ.get("SP_MODEL", "CohereLabs/tiny-aya-earth")
BASE_REV = os.environ.get("SP_MODEL_REVISION")
SOURCES = [s for s in os.environ.get("SP_SOURCES", "").split(",") if s]   # kernel slugs, for provenance
GIT_SHA = os.environ.get("SP_GIT_SHA")
TAG = os.environ.get("SP_HUB_TAG", "h1-v1")                         # version tag set after verification
ROOT = sp.out_dir("hub_stage")
shutil.rmtree(ROOT, ignore_errors=True)
ROOT.mkdir(parents=True)
REPOS = {}                                       # repo id -> {"type": model|dataset, "dir": staging dir}

def repo(suffix, rtype):
    rid = f"{OWNER}/{NAME}-{suffix}"
    REPOS.setdefault(rid, {"type": rtype, "dir": ROOT / f"{NAME}-{suffix}"})
    return REPOS[rid]["dir"]

def put(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(src, dst)

assert sp.hf_login(), "No HF_TOKEN: run this notebook from the Kaggle editor with the HF_TOKEN secret ticked."

No HF_TOKEN and no model mirror attached: gated models/datasets (Tiny Aya, Llama Guard, walledai/*) will not load. Attach `sp-models` or tick the HF_TOKEN secret.


AssertionError: No HF_TOKEN: run this notebook from the Kaggle editor with the HF_TOKEN secret ticked.

## Collect: adapters to model repos, everything else to dataset repos (harmful text skipped or stripped)

In [ ]:
# Model repos: LoRA adapters only, one repo per fine-tune condition.
runs = sp.list_runs()
for run, ckpts in runs.items():
    condition, _ = sp.parse_run(run)
    d = repo(f"lora-{condition}", "model")
    for step, ck in ckpts:
        for f in ("adapter_config.json", "adapter_model.safetensors"):
            put(ck / f, d / run / f"checkpoint-{step}" / f)

# Dataset repo: probe bundle, refusal direction, token sets (what `safetypulse check` needs).
pb = repo("probe-bundle", "dataset")
for name in ("probe_bundle.json", "directions.safetensors", "probe_pairs.jsonl"):
    hits = [h for h in sp.find_inputs(name) if h.parent.name == "probe_bundle"]
    if hits:
        put(hits[0], pb / name)
for name, sub in (("refusal_direction.json", "direction"), ("layer_selection.png", "direction"),
                  ("token_sets.json", "")):
    h = sp.find_input(name, required=False)
    if h:
        put(h, pb / sub / name)

# Dataset repo: logs and results.
lg = repo("logs", "dataset")
for run, ckpts in runs.items():
    log = ckpts[0][1].parent / "train_log.json"
    if log.exists():
        put(log, lg / "train_logs" / f"{run}.json")
rows = {}                                        # monitoring rows from every session; later rows win
for p in sp.find_inputs("monitor.jsonl"):
    for r in sp.load_jsonl(p):
        rows[(r["run"], r["step"])] = r
if rows:
    sp.save_jsonl(sorted(rows.values(), key=lambda r: (r["run"], r["step"])), lg / "monitor" / "monitor.jsonl")
br = sp.find_input("baseline_results.json", required=False)
if br:                                           # keep the numbers, drop example answers
    d = sp.load_json(br)
    for r in d.get("results", {}).values():
        for k in [k for k in r if k.startswith("examples")]:
            r.pop(k)
    sp.save_json(d, lg / "baseline" / "baseline_results.json")

for rid, r in REPOS.items():
    banned = [p for p in r["dir"].rglob("*") if p.name == "samples.json" or "circuit_validation" in str(p)]
    assert not banned, f"harmful-text files staged for {rid}: {banned}"
    if r["type"] == "model":
        extra = [p for p in r["dir"].rglob("*") if p.is_file() and p.name not in
                 ("adapter_config.json", "adapter_model.safetensors")]
        assert not extra, f"model repo {rid} would hold non-model files: {extra}"
print({k: len(v) for k, v in runs.items()}, "| monitor rows:", len(rows), "| repos:", list(REPOS))

## Manifest (SHA-256, sizes, provenance) and one card per repo

In [ ]:
def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

for rid, r in REPOS.items():
    r["files"] = {str(p.relative_to(r["dir"])): {"sha256": sha256(p), "bytes": p.stat().st_size}
                  for p in sorted(r["dir"].rglob("*")) if p.is_file()}
manifest = {
    "experiment": EXPERIMENT, "base_model": {"name": BASE, "revision": BASE_REV},
    "runs": {k: [s for s, _ in v] for k, v in runs.items()}, "monitor_rows": len(rows),
    "sources": {"kaggle_kernels": [f"https://www.kaggle.com/code/{s}" for s in SOURCES],
                "git_commit": GIT_SHA, "code": "https://github.com/PxA-Labs/safetypulse"},
    "withheld": "circuit-validation samples and example model answers (harmful text, docs/11)",
    "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "repos": {rid: {"type": r["type"], "n_files": len(r["files"]), "files": r["files"]} for rid, r in REPOS.items()},
}
sp.save_json(manifest, REPOS[f"{OWNER}/{NAME}-logs"]["dir"] / "MANIFEST.json")
REPOS[f"{OWNER}/{NAME}-logs"]["files"]["MANIFEST.json"] = None   # verified by size only (written after hashing)

LINKS = "**Code:** https://github.com/PxA-Labs/safetypulse" + (f" (commit `{GIT_SHA}`)" if GIT_SHA else "")
for rid, r in REPOS.items():
    kind = rid.split(f"{NAME}-", 1)[1]
    if r["type"] == "model":
        cond = kind.removeprefix("lora-")
        rs = sorted(k for k in runs if sp.parse_run(k)[0] == cond)
        head = (f"---\nlicense: cc-by-nc-4.0\nbase_model: {BASE}\nlibrary_name: peft\n"
                f"tags: [lora, safety, fine-tuning, safetypulse]\n---\n")
        body = (f"# SafetyPulse LoRA adapters on {BASE.split('/')[-1]}: {cond}\n\n"
                f"> **Research artifact, not for deployment.** Ordinary instruction tuning can weaken the base "
                f"model's refusals; these checkpoints exist to study and detect that drift.\n\n"
                f"Experiment: {EXPERIMENT}. Runs: {', '.join(f'`{k}` ({len(runs[k])} checkpoints)' for k in rs)}. "
                f"Base `{BASE}` (revision `{BASE_REV}`). LoRA r16 on all attention and MLP projections; "
                f"`checkpoint-0` is the untrained adapter (= base model). Each `<run>/checkpoint-<step>/` holds "
                f"`adapter_config.json` and `adapter_model.safetensors`. Checksums: `{OWNER}/{NAME}-logs` "
                f"`MANIFEST.json`.\n\n```python\nfrom transformers import AutoModelForCausalLM\n"
                f"from peft import PeftModel\nbase = AutoModelForCausalLM.from_pretrained(\"{BASE}\", "
                f"revision=\"{BASE_REV}\", torch_dtype=\"auto\")\nmodel = PeftModel.from_pretrained(base, "
                f"\"{rid}\", subfolder=\"{rs[0] if rs else '<run>'}/checkpoint-<step>\")\n```\n\n"
                f"Licence CC-BY-NC-4.0, inherited from the base model and the data. {LINKS}\n")
    else:
        head = "---\nlicense: cc-by-nc-4.0\ntags: [safety, safetypulse]\n---\n"
        what = {"probe-bundle": "the probe bundle `safetypulse check` needs (`probe_bundle.json`, "
                                "`directions.safetensors`, `probe_pairs.jsonl`), the refusal direction and "
                                "the model's refusal/compliance token sets",
                "logs": "training logs per run (`train_logs/`), CIS / spot-check / ASR for every checkpoint "
                        "(`monitor/monitor.jsonl`, no model answers), baseline numbers, and `MANIFEST.json` "
                        "(SHA-256 of every file in every repo of this experiment, with provenance)"}.get(kind, kind)
        body = (f"# SafetyPulse {kind} for {BASE.split('/')[-1]}\n\nExperiment: {EXPERIMENT}. Contains {what}. "
                f"Base `{BASE}` (revision `{BASE_REV}`). Withheld on purpose: {manifest['withheld']}. {LINKS}\n")
    (r["dir"] / "README.md").write_text(head + body)
    r["files"]["README.md"] = None
print(json.dumps({rid: len(r["files"]) for rid, r in REPOS.items()}, indent=1))

## Upload (resumable) and verify every file

In [ ]:
from huggingface_hub import HfApi
api = HfApi()

def upload(rid, rtype, folder):
    """Resumable where the library allows it; otherwise one commit per top-level entry (e.g. per run), each
    retried, so a dropped connection repeats only that part. Files already on the Hub are not re-sent."""
    if hasattr(api, "upload_large_folder"):          # huggingface_hub < 2.x
        return api.upload_large_folder(repo_id=rid, repo_type=rtype, folder_path=str(folder))
    for part in sorted(folder.iterdir()):
        for attempt in range(3):
            try:
                if part.is_dir():
                    api.upload_folder(repo_id=rid, repo_type=rtype, folder_path=str(part), path_in_repo=part.name,
                                      commit_message=f"add {part.name}")
                else:
                    api.upload_file(repo_id=rid, repo_type=rtype, path_or_fileobj=str(part),
                                    path_in_repo=part.name, commit_message=f"add {part.name}")
                break
            except Exception as e:
                if attempt == 2:
                    raise
                print(f"retrying {part.name} after {type(e).__name__}", flush=True)
                time.sleep(30)

receipt = {}
for rid, r in REPOS.items():
    api.create_repo(rid, repo_type=r["type"], private=True, exist_ok=True)
    assert api.repo_info(rid, repo_type=r["type"]).private, f"{rid} is public: refusing to upload a working backup"
    with sp.gpu_timer() as t:
        upload(rid, r["type"], r["dir"])
    remote = {e.path: e for e in api.list_repo_tree(rid, repo_type=r["type"], recursive=True, expand=True)
              if hasattr(e, "size")}
    missing = [f for f in r["files"] if f not in remote]
    bad = [f for f, m in r["files"].items() if m and f in remote and (remote[f].size != m["bytes"] or
           (getattr(remote[f], "lfs", None) and remote[f].lfs.sha256 != m["sha256"]))]
    assert not (missing or bad), {"repo": rid, "missing": missing, "mismatch": bad}
    commit = api.list_repo_commits(rid, repo_type=r["type"])[0].commit_id
    if TAG:   # citable version: the paper points at this tag, not at a moving main branch
        api.create_tag(rid, tag=TAG, revision=commit, repo_type=r["type"], exist_ok=True,
                       tag_message=f"{EXPERIMENT}; verified {len(r['files'])} files")
    receipt[rid] = {"type": r["type"], "verified_files": len(r["files"]), "commit": commit, "tag": TAG,
                    "minutes": round(t["seconds"] / 60, 1)}
    print("verified", rid, receipt[rid])
sp.save_json(receipt, ROOT / "upload_receipt.json")